# Notebook 2: Foreground Removal and Effects on Power Spectrum

Building on Notebook 1, this notebook:
1. Adds a synthetic foreground to the HI map
2. Analyses its frequency-frequency covariance and eigenstructure
3. Applies PCA-based foreground removal
4. Studies the effect on the 1D and cylindrical power spectra across 100 realisations

## 1. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm import tqdm
import time

from meer21cm import MockSimulation
from meer21cm.plot import plot_map
from meer21cm.fg import ForegroundSimulation
from meer21cm.util import pca_clean

## 2. Global Settings

In [ ]:
# Matplotlib font sizes used throughout the notebook
plt.rcParams.update({
    "font.size":        14,
    "axes.labelsize":   18,
    "axes.titlesize":   20,
    "xtick.labelsize":  14,
    "ytick.labelsize":  14,
    "legend.fontsize":  16,
})

# Seeds for the 100 realisations
seeds = np.arange(1, 101)

# Parallelism — use all available cores
num_processes = os.cpu_count()
print(f"Using {num_processes} parallel processes")

## 4. Function Definitions

All analysis and plotting functions are defined up front so the notebook can be run top-to-bottom.

### 4.0 Save figures 

In [ ]:
def save_project_plot(fig, week_number, filename):
    """
    Save a matplotlib figure into the project outputs directory.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
    week_number : int
    filename : str
    """
    base_path = "/Users/gracetait/shproject/senior_honours_project/outputs"
    target_dir = os.path.join(base_path, f"week{week_number}")

    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
        print(f"Created new directory: {target_dir}")

    save_path = os.path.join(target_dir, filename)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Successfully saved: {save_path}")

### 4.1 Data generation — single realisation

In [ ]:
def generate_power_spectrum_data(seed, fg_map, k_perppara_min=None, k_perppara_max=None, map_type="hi"):
    """
    Generate mock HI data for a given seed, apply PCA foreground removal,
    and return 1D and cylindrical power spectra.

    Parameters
    ----------
    seed : int
    fg_map : ndarray
        Pre-computed foreground map (generated once; does not vary with seed).
    k_perppara_min, k_perppara_max : list or None
        k-mode thresholds passed to ``get_1d_power``.
    map_type : str
        One of 'hi', 'fg', or 'noise'.

    Returns
    -------
    k_1d, power_1d_map, power_1d_res, power_cy_map, power_cy_res
    """
    if not isinstance(seed, (int, np.integer)):
        print("Error: seed must be an integer.")
        return

    mock = MockSimulation(
        survey="meerklass_2021",
        band="L",
        ra_range=(334, 357),
        dec_range=(-35, -26.5),
        flat_sky=True,
        seed=seed,
        omega_hi=5e-4,
        mean_amp_1="average_hi_temp",
        tracer_bias_1=1.0,
    )

    hi_map = mock.propagate_mock_field_to_data(mock.mock_tracer_field_1)

    sigma_noise = 1.21e-3
    noise_map = np.random.default_rng(mock.seed).normal(0, sigma_noise, hi_map.shape) * mock.w_HI

    if map_type == "hi":
        map_data = hi_map
    elif map_type == "fg":
        map_data = fg_map
    elif map_type == "noise":
        map_data = noise_map
    else:
        print(f"Invalid map_type '{map_type}'. Choose from 'hi', 'fg', 'noise'.")
        return

    tot_map = hi_map + fg_map
    res_map = pca_clean(tot_map, 3, weights=mock.w_HI, return_analysis=False, mean_center=True)

    mock.kparabins = np.linspace(0.01, 1, 11)
    mock.kperpbins = np.linspace(0.01, 0.5, 21)
    mock.k1dbins   = np.linspace(0, 1.2, 21)

    mock.data = map_data
    mock.grid_data_to_field()
    power_cy_map, _         = mock.get_cy_power(mock.auto_power_3d_1)
    power_1d_map, k_1d, _  = mock.get_1d_power(mock.auto_power_3d_1)

    mock.data = res_map
    mock.grid_data_to_field()
    power_cy_res, _         = mock.get_cy_power(mock.auto_power_3d_1)
    power_1d_res, k_1d, _  = mock.get_1d_power(
        mock.auto_power_3d_1,
        k_perppara_min=k_perppara_min,
        k_perppara_max=k_perppara_max,
    )

    return k_1d, power_1d_map, power_1d_res, power_cy_map, power_cy_res

### 4.2 Data generation — 100 realisations (parallel)

In [ ]:
def generate_power_spectrum_data_100_realisations(fg_map, k_perppara_min=None, k_perppara_max=None, map_type="hi"):
    """
    Run ``generate_power_spectrum_data`` for all seeds in parallel.

    Returns lists of arrays, one entry per seed.
    """
    start = time.time()

    results = Parallel(n_jobs=num_processes)(
        delayed(generate_power_spectrum_data)(seed, fg_map, k_perppara_min, k_perppara_max, map_type)
        for seed in tqdm(seeds, desc="Processing seeds")
    )

    elapsed = time.time() - start
    print(f"Total time: {elapsed / 60:.2f} min ({elapsed:.1f} s)")

    k_1d_list          = [r[0] for r in results]
    power_1d_hi_list   = [r[1] for r in results]
    power_1d_res_list  = [r[2] for r in results]
    power_cy_hi_list   = [r[3] for r in results]
    power_cy_res_list  = [r[4] for r in results]

    return k_1d_list, power_1d_hi_list, power_1d_res_list, power_cy_hi_list, power_cy_res_list

### 4.3 Plotting — 1D log power spectrum

In [ ]:
def plot_1d_log_power_spectrum(k_1d_list, power_1d_hi_list, power_1d_res_list,
                                k_perppara_min=None, k_perppara_max=None,
                                errors=True, map_type="hi",
                                save_plot=False, week_number=None):
    """Plot the log10 1D power spectrum, before and after foreground removal."""
    fs = 14

    log_hi  = np.log10(power_1d_hi_list)
    log_res = np.log10(power_1d_res_list)

    k_mean     = np.mean(k_1d_list, axis=0)
    k_std      = np.std(k_1d_list, axis=0)
    hi_mean    = np.mean(log_hi,  axis=0)
    hi_std     = np.abs(np.std(log_hi,  axis=0))
    res_mean   = np.mean(log_res, axis=0)
    res_std    = np.abs(np.std(log_res, axis=0))

    plt.plot(k_mean, hi_mean,  label="HI signal",         color="blue", linewidth=1)
    plt.plot(k_mean, res_mean, label="Residual signal",    color="red",  linewidth=1, ls="--")

    if errors:
        plt.errorbar(k_mean, hi_mean,  yerr=hi_std,  xerr=k_std,
                     fmt="o",    capsize=3, color="blue", markersize=1, linewidth=1)
        plt.errorbar(k_mean, res_mean, yerr=res_std, xerr=k_std,
                     fmt="None", capsize=3, color="red",  markersize=1, linewidth=1)

    plt.xlabel(r"k [Mpc$^{-1}$]",                              fontsize=fs)
    plt.ylabel(r"log$_{10}$ P(k) [${\rm Mpc}^{3}K^2]$)",     fontsize=fs)
    plt.title("1D Power Spectrum",                              fontsize=fs + 2)
    plt.tick_params(labelsize=fs - 2)
    plt.legend(fontsize=fs)

    if save_plot:
        save_project_plot(plt.gcf(), week_number=week_number,
                          filename=f"1d_log_power_spectrum_{k_perppara_min}_{k_perppara_max}_{map_type}.pdf")
    plt.show()

### 4.4 Plotting — 1D power spectrum (k^{3/2} scaled)

In [ ]:
def plot_1d_power_spectrum(k_1d_list, power_1d_hi_list, power_1d_res_list,
                            k_perppara_min=None, k_perppara_max=None,
                            errors=True, map_type="hi",
                            save_plot=False, week_number=None):
    """Plot P(k) * k^{3/2}, before and after foreground removal."""
    fs = 14

    k   = np.array(k_1d_list)
    hi  = np.array(power_1d_hi_list)
    res = np.array(power_1d_res_list)

    y_hi  = hi  * k ** (3 / 2)
    y_res = res * k ** (3 / 2)

    k_mean    = np.mean(k, axis=0);        k_std    = np.std(k, axis=0)
    hi_mean   = np.mean(y_hi,  axis=0);    hi_std   = np.std(y_hi,  axis=0)
    res_mean  = np.mean(y_res, axis=0);    res_std  = np.std(y_res, axis=0)

    plt.plot(k_mean, hi_mean,  label="no foreground removal")
    plt.plot(k_mean, res_mean, label="after foreground removal", ls="--")

    if errors:
        plt.errorbar(k_mean, hi_mean,  yerr=hi_std,  xerr=k_std,
                     fmt="o",    capsize=3, color="blue", markersize=1, linewidth=1)
        plt.errorbar(k_mean, res_mean, yerr=res_std, xerr=k_std,
                     fmt="None", capsize=3, color="red",  markersize=1, linewidth=1)

    plt.xlabel(r"k [Mpc$^{-1}$]",                                  fontsize=fs)
    plt.ylabel(r"P(k)$k^{3/2}$ [${\rm Mpc}^{3/2}K^2]$)",         fontsize=fs)
    plt.title("1D Power Spectrum",                                  fontsize=fs + 2)
    plt.tick_params(labelsize=fs - 2)
    plt.legend()

    if save_plot:
        save_project_plot(plt.gcf(), week_number=week_number,
                          filename=f"1d_power_spectrum_{k_perppara_min}_{k_perppara_max}_{map_type}.pdf")
    plt.show()

### 4.5 Plotting — cylindrical (2D) power spectrum

In [ ]:
def plot_cylindrical_power_spectrum(power_cy_hi_list, power_cy_res_list,
                                     k_perppara_min=None, k_perppara_max=None,
                                     map_type="hi", save_plot=False, week_number=None):
    """
    Three-panel cylindrical power spectrum plot:
    left = signal, centre = residual after cleaning, right = ratio.
    """
    fs = 14

    kperpbins = np.linspace(0.01, 0.5, 21)
    kparabins = np.linspace(0.01, 1,   11)

    hi_arr  = np.array(power_cy_hi_list)
    res_arr = np.array(power_cy_res_list)
    hi_mean  = np.mean(hi_arr,  axis=0)
    res_mean = np.mean(res_arr, axis=0)

    log_hi  = np.log10(hi_mean.T)
    log_res = np.log10(res_mean.T)
    vmin = min(log_hi.min(), log_res.min())
    vmax = max(log_hi.max(), log_res.max())

    titles = {"hi": "HI power spectrum", "fg": "Foreground", "noise": "Noise"}
    if map_type not in titles:
        print(f"Invalid map_type '{map_type}'.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].pcolormesh(kperpbins, kparabins, log_hi, vmin=vmin, vmax=vmax)
    axes[0].set_xlabel(r"k$_\perp$ [Mpc$^{-1}$]", fontsize=fs)
    axes[0].set_ylabel(r"k$_\parallel$ [Mpc$^{-1}$]", fontsize=fs)
    axes[0].set_title(titles[map_type], fontsize=fs + 2)
    axes[0].tick_params(labelsize=fs - 2)

    im = axes[1].pcolormesh(kperpbins, kparabins, log_res, vmin=vmin, vmax=vmax)
    axes[1].set_xlabel(r"k$_\perp$ [Mpc$^{-1}$]", fontsize=fs)
    axes[1].set_ylabel(r"k$_\parallel$ [Mpc$^{-1}$]", fontsize=fs)
    axes[1].set_title("Residual signal", fontsize=fs + 2)
    axes[1].tick_params(labelsize=fs - 2)

    cbar1 = plt.colorbar(im, ax=axes[:2], location="top", aspect=50, pad=0.1)
    cbar1.set_label(r"log$_{10}$ P(k$_\perp$, k$_\parallel$) [${\rm Mpc}^{3}K^2]$)", fontsize=fs)
    cbar1.ax.tick_params(labelsize=fs - 2)

    im2 = axes[2].pcolormesh(kperpbins, kparabins, res_mean.T / hi_mean.T, cmap="bwr")
    axes[2].set_xlabel(r"k$_\perp$ [Mpc$^{-1}$]", fontsize=fs)
    axes[2].set_ylabel(r"k$_\parallel$ [Mpc$^{-1}$]", fontsize=fs)
    axes[2].set_title("Ratio (residual / signal)", fontsize=fs + 1)
    axes[2].tick_params(labelsize=fs - 2)
    cbar2 = plt.colorbar(im2, ax=axes[2], location="top", aspect=23, pad=0.1)
    cbar2.ax.tick_params(labelsize=fs - 2)

    if save_plot:
        save_project_plot(plt.gcf(), week_number=week_number,
                          filename=f"cylindrical_power_spectrum_{k_perppara_min}_{k_perppara_max}_{map_type}.pdf")
    plt.show()

### 4.6 Combined foreground removal analysis (variable number of modes)

In [ ]:
def foreground_removal_analysis(hi_map, fg_map, mock, modes_cleaned=3,
                                 save_plot=False, week_number=None):
    """
    Run PCA cleaning with a specified number of modes removed, then plot:
    - 1D power spectrum (k^{3/2} and log)
    - Residuals
    - Cylindrical power spectrum

    Parameters
    ----------
    hi_map, fg_map : ndarray
    mock : MockSimulation  (used for k-bins and projection)
    modes_cleaned : int
    save_plot : bool
    week_number : int or None
    """
    fs = 14
    method = f"pca_{modes_cleaned}modes"

    tot_map = hi_map + fg_map
    res_map = pca_clean(tot_map, modes_cleaned, weights=mock.w_HI,
                        return_analysis=False, mean_center=True)

    mock.kparabins = np.linspace(0.01, 1, 11)
    mock.kperpbins = np.linspace(0.01, 0.5, 21)
    mock.k1dbins   = np.linspace(0, 1.2, 21)

    mock.data = hi_map;  mock.grid_data_to_field()
    power_cy_hi,  _ = mock.get_cy_power(mock.auto_power_3d_1)
    power_1d_hi, k_1d, _ = mock.get_1d_power(mock.auto_power_3d_1)

    mock.data = res_map; mock.grid_data_to_field()
    power_cy_res, _ = mock.get_cy_power(mock.auto_power_3d_1)
    power_1d_res, k_1d, _ = mock.get_1d_power(mock.auto_power_3d_1)

    # ── 1D power spectrum (k^{3/2} scaled) ──
    y_hi  = (power_1d_hi.T)  * k_1d ** (3 / 2)
    y_res = (power_1d_res.T) * k_1d ** (3 / 2)

    plt.plot(k_1d, y_hi,  label="no FG removal",     color="blue")
    plt.plot(k_1d, y_res, label="after FG removal",   color="red", ls="--")
    plt.xlabel(r"k [Mpc$^{-1}$]");  plt.ylabel(r"P(k)$k^{3/2}$ [Mpc$^{3/2}$K$^2$]")
    plt.title(f"1D Power Spectrum — {modes_cleaned} modes removed")
    plt.legend()
    if save_plot:
        save_project_plot(plt.gcf(), week_number=week_number,
                          filename=f"1d_power_spectrum_{method}.pdf")
    plt.show()

    # ── Residuals ──
    plt.plot(k_1d, y_res - y_hi, color="red", ls="--", label="residual (after − before)")
    plt.axhline(0, color="blue", linewidth=0.8)
    plt.xlabel(r"k [Mpc$^{-1}$]");  plt.ylabel(r"$\Delta$P(k)$k^{3/2}$ [Mpc$^{3/2}$K$^2$]")
    plt.title("Signal Loss");  plt.legend()
    if save_plot:
        save_project_plot(plt.gcf(), week_number=week_number,
                          filename=f"1d_power_spectrum_residuals_{method}.pdf")
    plt.show()

    # ── Log 1D power spectrum ──
    log_hi  = np.log10(power_1d_hi)
    log_res = np.log10(power_1d_res)
    plt.plot(k_1d, log_hi,  label="no FG removal",   color="blue", linewidth=1)
    plt.plot(k_1d, log_res, label="after FG removal", color="red",  linewidth=1, ls="--")
    plt.xlabel(r"k [Mpc$^{-1}$]")
    plt.ylabel(r"log$_{10}$ P(k) [${\rm Mpc}^{3}K^2]$)")
    plt.title(f"Log 1D Power Spectrum — {modes_cleaned} modes removed")
    plt.legend()
    if save_plot:
        save_project_plot(plt.gcf(), week_number=week_number,
                          filename=f"log_1d_power_spectrum_{method}.pdf")
    plt.show()

    # ── Cylindrical power spectrum ──
    vmin = np.log10([power_cy_hi.min(), power_cy_res.min()]).min()
    vmax = np.log10([power_cy_hi.max(), power_cy_res.max()]).max()
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].pcolormesh(mock.kperpbins, mock.kparabins, np.log10(power_cy_hi.T),  vmin=vmin, vmax=vmax)
    axes[0].set_xlabel(r"k$_\perp$ [Mpc$^{-1}$]", fontsize=fs)
    axes[0].set_ylabel(r"k$_\parallel$ [Mpc$^{-1}$]", fontsize=fs)
    axes[0].set_title("HI signal", fontsize=fs + 2)
    axes[0].tick_params(labelsize=fs - 2)
    im = axes[1].pcolormesh(mock.kperpbins, mock.kparabins, np.log10(power_cy_res.T), vmin=vmin, vmax=vmax)
    axes[1].set_xlabel(r"k$_\perp$ [Mpc$^{-1}$]", fontsize=fs)
    axes[1].set_ylabel(r"k$_\parallel$ [Mpc$^{-1}$]", fontsize=fs)
    axes[1].set_title("Residual signal", fontsize=fs + 2)
    axes[1].tick_params(labelsize=fs - 2)
    axes[1].set_yticklabels([]); axes[1].set_yticks([])
    cbar1 = plt.colorbar(im, ax=axes[:2], location="top", aspect=50, pad=0.1)
    cbar1.set_label(r"log$_{10}$ P(k$_\perp$, k$_\parallel$) [${\rm Mpc}^{3}K^2]$)", fontsize=fs)
    cbar1.ax.tick_params(labelsize=fs - 2)
    im2 = axes[2].pcolormesh(mock.kperpbins, mock.kparabins,
                              power_cy_res.T / power_cy_hi.T, cmap="bwr")
    axes[2].set_xlabel(r"k$_\perp$ [Mpc$^{-1}$]", fontsize=fs)
    axes[2].set_ylabel(r"k$_\parallel$ [Mpc$^{-1}$]", fontsize=fs)
    axes[2].set_title("Ratio", fontsize=fs + 2)
    axes[2].tick_params(labelsize=fs - 2)
    axes[2].set_yticklabels([]); axes[2].set_yticks([])
    cbar2 = plt.colorbar(im2, ax=axes[2], location="top", aspect=23, pad=0.1)
    cbar2.ax.tick_params(labelsize=fs - 2)
    if save_plot:
        save_project_plot(plt.gcf(), week_number=week_number,
                          filename=f"cylindrical_power_spectrum_{method}.pdf")
    plt.show()

## 5. Simulation Setup

Create the base mock simulation and compute the foreground map. The foreground simulation is slow but only needs to run once since it does not depend on the seed.

In [ ]:
mock = MockSimulation(
    survey="meerklass_2021",
    band="L",
    ra_range=(334, 357),
    dec_range=(-35, -26.5),
    flat_sky=True,
    seed=42,
    omega_hi=5e-4,
    mean_amp_1="average_hi_temp",
    tracer_bias_1=1.0,
)

fgsim = ForegroundSimulation(
    hp_nside=128,
    wproj=mock.wproj,
    num_pix_x=mock.num_pix_x,
    num_pix_y=mock.num_pix_y,
    backend="pysm",
)

hi_map = mock.propagate_mock_field_to_data(mock.mock_tracer_field_1)
fg_map = fgsim.fg_wcs_cube(mock.nu)  # slow — save/load if re-running

## 6. Visualising the Maps

In [ ]:
plot_map(hi_map, mock.wproj, W=mock.W_HI, title="HI map")
save_project_plot(plt.gcf(), week_number=3, filename="hydrogen_map_wo_foreground.pdf")
plt.show()

In [ ]:
plot_map(fg_map, mock.wproj, W=mock.W_HI, title="Foreground map")
save_project_plot(plt.gcf(), week_number=3, filename="hydrogen_map_w_foreground.pdf")
plt.show()

The foreground is ~4 orders of magnitude brighter than the HI signal, but has a very different frequency structure — it is smooth in frequency, whereas the HI signal fluctuates rapidly.

In [ ]:
fs = 16
plt.plot(mock.nu / 1e6, fg_map[60, 35], label="Foreground signal")
plt.plot(mock.nu / 1e6, hi_map[60, 35] * 1e3, label=r"HI signal $\times 10^3$")
plt.xlabel("Frequency [MHz]", fontsize=fs)
plt.ylabel("Temperature [K]", fontsize=fs)
plt.title("Temperature vs Frequency", fontsize=fs + 2, pad=10)
plt.legend(fontsize=fs)
save_project_plot(plt.gcf(), week_number=3, filename="frequency_vs_temperature.pdf")
plt.show()

## 7. Frequency-Frequency Covariance and PCA

The covariance matrix captures correlations between frequency channels, which foregrounds dominate.

In [ ]:
tot_map = hi_map + fg_map

cov_tot, _, eigen_values, eigen_vectors = pca_clean(
    tot_map, 1, weights=mock.w_HI, return_analysis=True, mean_center=True
)
cov_fg, _, _, _ = pca_clean(
    fg_map, 1, weights=mock.w_HI, return_analysis=True, mean_center=True
)
cov_hi, _, _, _ = pca_clean(
    hi_map, 1, weights=mock.w_HI, return_analysis=True, mean_center=True
)

In [ ]:
# plot the covariance of the HI, foreground, and noise signals
fig, axes = plt.subplots(1, 3, figsize=(15, 8), layout="constrained")

# Define shared parameters
extent = [mock.nu[0]/1e6, mock.nu[-1]/1e6, mock.nu[0]/1e6, mock.nu[-1]/1e6]
titles = ['Total signal', 'Foreground signal', 'HI signal']
covs = [cov_tot, cov_fg, cov_hi]

fs = 16

for i, (ax, cov, title) in enumerate(zip(axes, covs, titles)):
    im = ax.imshow(cov, origin='lower', extent=extent)
    
    # Add colorbar to each individual axis
    cb = fig.colorbar(im, ax=ax, location='right', fraction=0.05, pad=0.04)
    cb.ax.tick_params(labelsize=fs-2)
    cb.ax.yaxis.get_offset_text().set_fontsize(fs-2)
    
    # X-axis is the same for all
    ax.set_xlabel('Frequency [MHz]', fontsize=fs)
    ax.set_title(title, fontsize=fs+2, pad = 10)
    ax.tick_params(axis='x', labelsize=fs-2)

    # Y-axis logic: Only keep for the first plot
    if i == 0:
        ax.set_ylabel('Frequency [MHz]', fontsize=fs)
        ax.tick_params(axis='y', labelsize=fs-2)
    else:
        # This removes both the tick marks and the numbers
        ax.set_yticklabels([]) 
        ax.set_yticks([])

save_project_plot(plt.gcf(), week_number=9, filename="tot_fg_hi_covariance.pdf")
plt.show()

## 8. Eigenvalue Spectrum

The foreground modes produce distinctly large eigenvalues; the signal and noise plateau at a much lower level.

In [ ]:
fs = 16
plt.figure(figsize=(8, 5))
plt.plot(eigen_values, marker="o", markersize=4, color="black", linewidth=1)
plt.yscale("log")
plt.xlim(-0.5, 20)
plt.xlabel("Eigenvalue Index", fontsize=fs)
plt.ylabel("Eigenvalue", fontsize=fs)
plt.title("Eigenvalue vs Index", fontsize=fs + 2, pad=15)
plt.tick_params(axis="both", which="major", labelsize=fs)
save_project_plot(plt.gcf(), week_number=4, filename="eigenvalues.pdf")
plt.show()

The first 3 eigenvalues (indices 0, 1, 2) are orders of magnitude larger than the rest — these correspond to the foreground modes.

## 9. Eigenvectors

In [ ]:
fs = 22
fig, axes = plt.subplots(10, 1, figsize=(10, 12), sharex=True, layout="constrained")
for i in range(10):
    axes[i].plot(mock.nu / 1e6, eigen_vectors[:, i], color="black")
    axes[i].tick_params(axis="both", which="major", labelsize=fs - 2)

fig.supylabel("Eigenvector", fontsize=fs)
axes[-1].set_xlabel("Frequency [MHz]", fontsize=fs)
axes[0].set_title("Eigenvectors vs Frequency", fontsize=fs + 2, pad=10)
save_project_plot(plt.gcf(), week_number=4, filename="eigenvectors.pdf")
plt.show()

## 10. Effect of the Number of Modes Removed

Removing too few modes leaves foreground residuals; removing too many causes signal loss. Here we sweep over 0–3 modes.

In [ ]:
for n_modes in [0, 1, 2, 3]:
    print(f"\n{'='*50}")
    print(f"  Removing {n_modes} modes")
    print(f"{'='*50}")
    foreground_removal_analysis(hi_map, fg_map, mock, modes_cleaned=n_modes,
                                 save_plot=False)

## 11. Power Spectra for HI, Foreground, and Noise Signals

Using a single realisation (seed=42) to examine how each signal component looks in the power spectrum.

In [ ]:
for map_type in ["hi", "fg", "noise"]:
    print(f"\nMap type: {map_type}")
    k_1d, power_1d_map, power_1d_res, power_cy_map, power_cy_res = generate_power_spectrum_data(
        42, fg_map, k_perppara_min=None, k_perppara_max=None, map_type=map_type
    )
    plot_cylindrical_power_spectrum(
        [power_cy_map], [power_cy_res],
        map_type=map_type, save_plot=False
    )

## 12. k-Mode Threshold Sweep (100 Realisations)

Signal loss after foreground removal occurs mainly at low $k_\parallel$. By cutting out contaminated k-modes (setting `k_perppara_min`) we can recover better agreement between cleaned and original power spectra.

The cell below sweeps over a range of thresholds and plots each result.

In [ ]:
# k_perppara_min values to explore: [k_perp_min, k_para_min]
thresholds_to_test = [
    None,
    [0,     0.01],
    [0,     0.02],
    [0,     0.03],
    [0,     0.04],
    [0,     0.05],
    [0,     0.06],
    [0,     0.08],
    [0,     0.10],
    [0.015, 0.06],
    [0.02,  0.06],
    [0.025, 0.06],
    [0.025, 0.08],
    [0.025, 0.10],
]

for threshold in thresholds_to_test:
    print(f"\nk_perppara_min = {threshold}")
    k_1d_list, power_1d_hi_list, power_1d_res_list, power_cy_hi_list, power_cy_res_list = \
        generate_power_spectrum_data_100_realisations(
            fg_map, k_perppara_min=threshold, k_perppara_max=None
        )
    plot_1d_log_power_spectrum(k_1d_list, power_1d_hi_list, power_1d_res_list,
                                k_perppara_min=threshold, k_perppara_max=None,
                                errors=True, save_plot=False)

## 13. Best-Threshold Comparison

Compare the best-performing threshold against the uncleaned case using 100 realisations.

In [ ]:
# Run with no threshold (baseline)
k_1d_list, power_1d_hi_list, power_1d_res_list, power_cy_hi_list, power_cy_res_list = \
    generate_power_spectrum_data_100_realisations(fg_map, k_perppara_min=None, k_perppara_max=None)

# Run with the best threshold identified above — update [0, 0.03] as needed
best_threshold = [0, 0.03]
k_1d_list_c, power_1d_hi_list_c, power_1d_res_list_c, power_cy_hi_list_c, power_cy_res_list_c = \
    generate_power_spectrum_data_100_realisations(fg_map, k_perppara_min=best_threshold, k_perppara_max=None)

In [ ]:
fs = 16

log_hi    = np.log10(power_1d_hi_list)
log_res   = np.log10(power_1d_res_list)
log_res_c = np.log10(power_1d_res_list_c)

k_mean     = np.mean(k_1d_list, axis=0);    k_std     = np.std(k_1d_list, axis=0)
k_mean_c   = np.mean(k_1d_list_c, axis=0);  k_std_c   = np.std(k_1d_list_c, axis=0)
hi_mean    = np.mean(log_hi,    axis=0);     hi_std    = np.abs(np.std(log_hi,    axis=0))
res_mean   = np.mean(log_res,   axis=0);     res_std   = np.abs(np.std(log_res,   axis=0))
res_mean_c = np.mean(log_res_c, axis=0);     res_std_c = np.abs(np.std(log_res_c, axis=0))

plt.plot(k_mean,   hi_mean,    label="HI signal",                     color="black", linewidth=1)
plt.plot(k_mean,   res_mean,   label="Residual (no threshold)",        color="blue",  linewidth=1, ls="--")
plt.plot(k_mean_c, res_mean_c, label=f"Residual (threshold {best_threshold})", color="red", linewidth=1, ls="--")

plt.errorbar(k_mean,   hi_mean,    yerr=hi_std,    xerr=k_std,   fmt="o",    capsize=3, color="black", markersize=1, linewidth=1)
plt.errorbar(k_mean,   res_mean,   yerr=res_std,   xerr=k_std,   fmt="None", capsize=3, color="blue",  markersize=1, linewidth=1)
plt.errorbar(k_mean_c, res_mean_c, yerr=res_std_c, xerr=k_std_c, fmt="None", capsize=3, color="red",   markersize=1, linewidth=1)

plt.xlabel(r"k [Mpc$^{-1}$]",                           fontsize=fs)
plt.ylabel(r"log$_{10}$ P(k) [${\rm Mpc}^{3}K^2]$)",  fontsize=fs)
plt.title("1D Power Spectrum — Threshold Comparison",   fontsize=fs + 2)
plt.tick_params(labelsize=fs - 2)
plt.legend(fontsize=fs - 2)
save_project_plot(plt.gcf(), week_number=9,
                  filename=f"1d_log_power_spectrum_comparison_{best_threshold}.pdf")
plt.show()